# 3. O Processo de Modelagem

Notas do curso **Machine Learning Process** — passo a passo do processo de modelagem, da definição do problema até colocar o modelo em produção.

---

### Índice
1. [Visão geral do pipeline](#pipeline)
2. [Enquadramento do problema](#framing)
3. [Coleta de dados](#coleta)
4. [Pré-processamento](#pre)
5. [EDA](#eda)
6. [Feature Engineering](#features)
7. [Modelagem](#modelagem)
8. [Validação cruzada](#validacao)
9. [Avaliação](#avaliacao)
10. [Produtização](#producao)
11. [Templates e resumo interativo](#templates)

---
<a id='pipeline'></a>
## 1. Visão geral do pipeline

```
  Enquadramento → Coleta → Pré-processamento → EDA → Feature engineering
       → Modelagem → Validação → Avaliação → Produtização
            ↑_________________________________________|
                              (iterar)
```

Cada etapa alimenta a próxima. O ciclo nunca é linear — você volta às etapas anteriores conforme aprende com os dados e com o negócio.

---
<a id='framing'></a>
## 2. Enquadramento do problema (*Problem Framing*)

**Primeiro passo crítico:** transformar um problema vago em um problema definido, com escopo, métrica e critério de sucesso. **Não pular direto para dados e modelos.**

> ML só entra quando o problema **definido** se beneficia de predição — não quando o problema ainda é "algo está errado".

### Perguntas essenciais

| Pergunta | Por que importa |
|---|---|
| **Quem é o usuário final?** | Define UX da solução e como medir sucesso |
| **Qual problema estamos resolvendo?** | Uma frase específica — não "usar IA" |
| **Qual é o impacto?** | Receita, custo, risco — quantificar se possível |
| **Qual é a urgência?** | Prazo, janela de campanha — prioriza simples vs. ML completo |

### Escala, restrições e dependências

**Escala**
- Quantos usuários/eventos/linhas por dia?
- Batch (diário) ou tempo real (milissegundos)?
- Crescimento esperado nos próximos 12 meses?

**Restrições**
- Orçamento, prazo, equipe (ML, eng, produto)
- Privacidade / LGPD
- Interpretabilidade obrigatória vs. máxima acurácia

**Dependências**
- De quais times ou sistemas dependemos?
- O que bloqueia o projeto se não for entregue primeiro?

### Problema bem definido

| Critério | O que significa |
|---|---|
| **Quantitativo** | Métricas numéricas de sucesso (%, R$, tempo) |
| **Específico** | Quem, o quê, quando, onde — sem ambiguidade |
| **Focado no usuário** | Benefício para quem usa o produto |
| **Com restrições** | Limites explícitos — muitas vezes baseados em tempo |

### Impacto de negócio

| Tipo | O que mede | Exemplos |
|---|---|---|
| **Accuracy score** | Performance técnica | AUC, F1, RMSE |
| **Business score** | Impacto real no KPI | Retenção D30, receita incremental |

**Como pensar o valor do projeto:**
1. **Baseline** — o que acontece *sem* a solução nova?
2. **Lift esperado** — se acerta X% a mais, quantos usuários/reais move?
3. **Conversão em valor** — clique → compra → margem
4. **Custo do projeto** — tempo de ML + eng + infra vs. ganho

### Matriz esforço × impacto

```
        impacto alto
             │
   quick     │    strategic
   wins      │    (ML full)
             │
        ─────┼───── esforço alto
             │
   evitar    │    reconsiderar
             │
        impacto baixo
```

Priorize soluções no quadrante **alto impacto, baixo esforço** antes de um pipeline ML longo.

---
<a id='coleta'></a>
## 3. Coleta de dados

Após o *data audit* do enquadramento, execute e documente a coleta.

> Detalhes das fontes: veja o notebook **4. Data Collection**

| Pergunta | Ação |
|---|---|
| Que dados endereçam o problema? | Listar tabelas, pipelines, APIs |
| Estado aceitável? | Critérios de qualidade mínima acordados |
| Sinais suficientes? | Gap analysis de features vs. alvo |
| Coletar mais? | Tickets para eng/analytics, novos eventos no app |

---
<a id='pre'></a>
## 4. Pré-processamento

Tratar nulos, outliers, tipos e consistência entre fontes.

> Detalhes: veja o notebook **5. Data Preprocessing**

| Tarefa | O que fazer |
|---|---|
| **Valores nulos** | Identificar missing, decidir: remover, imputar ou criar flag |
| **Outliers** | IQR, z-score, regras de negócio — às vezes são o sinal (whales) |
| **Tipos e formatos** | Datas, categorias, IDs — tipos corretos, sem duplicatas inconsistentes |
| **Consistência** | Mesma métrica definida igual em todas as fontes |

> Dados sujos no pré-processamento → EDA enganosa → modelo errado → hipóteses falsas.

---
<a id='eda'></a>
## 5. Análise Exploratória de Dados (EDA)

Entender **o que os dados mostram** antes de assumir causas.

- Distribuições (histogramas, box plots)
- Tendências no tempo (retenção, conversão por cohort)
- Relações entre variáveis (correlação, segmentos)
- Comparar grupos (quem converte vs. quem não converte?)
- Verificar **leakage** (features que revelam o futuro)
- Validar se há **sinal** antes de modelar caro

**Objetivo:** perguntas melhores e suspeitas sobre *onde* investigar — não pular direto para ML.

---
<a id='features'></a>
## 6. Feature Engineering

Transformar dados brutos em **sinais** que o modelo consome.

**Exemplos em produto:**
- Dias desde o cadastro, sessões na última semana
- Taxa de clique em e-mails, features de funil
- Agregações por usuário / produto / campanha
- Encoding de categorias (país, plano, canal)
- Features de lag, rolling windows (forecasting)
- Embeddings (recomendação)

**Regra:** boas features ligam comportamento do usuário ao **KPI** que o negócio quer mover.

**Atenção ao momento da predição:** a feature precisa estar disponível em produção *no momento* em que o modelo vai prever — sem usar dados do futuro.

---
<a id='modelagem'></a>
## 7. Modelagem

### Como escolher

| Pergunta | Guia |
|---|---|
| **Que tipo de algoritmo?** | Testar baseline simples primeiro, depois candidatos mais complexos |
| **Supervisionado ou não?** | Tem rótulo → supervisionado. Só grupos → não supervisionado |
| **Classificação ou regressão?** | Alvo categórico → classificação. Alvo contínuo → regressão |

### Classificação vs. Regressão — prós e contras

| | **Classificação** | **Regressão** |
|---|---|---|
| **Prós** | Decisão clara; fácil de comunicar | Estima magnitude; útil para priorização fina |
| **Contras** | Perde granularidade se o negócio precisa de "quanto" | Erros em escala podem distorcer ROI |

### Estratégia
1. Definir **baseline** (regra simples, média, modelo legado)
2. Testar candidatos progressivamente mais complexos
3. Comparar com métricas alinhadas ao KPI — não só acurácia técnica
4. Só parar quando houver um **modelo vencedor** claro vs. baseline

---
<a id='validacao'></a>
## 8. Validação Cruzada (*Cross Validation*)

**Objetivo:** evitar overfitting e estimar performance realista antes de produção.

| Técnica | Quando usar |
|---|---|
| **K-Fold** | Dados i.i.d. (cada amostra é independente) |
| **Validação temporal** | Forecasting, recomendação — não embaralhar o tempo! |

**Splits obrigatórios:** treino / validação / teste — com critério documentado.

> ⚠️ Em séries temporais: sempre treinar em dados passados e validar em dados futuros. Jamais embaralhar aleatoriamente.

---
<a id='avaliacao'></a>
## 9. Avaliação (*Evaluation*)

| Contexto | O que medir |
|---|---|
| **Offline** | Métricas técnicas (AUC, F1, RMSE) + proxy de negócio |
| **Online** | A/B test, holdout, impacto no KPI do produto |
| **Produto** | Latência, disponibilidade, fairness |

> ⚠️ Não confundir **boa métrica offline** com **sucesso no produto**. Um modelo com AUC 0.95 que não move retenção não vale o custo.

---
<a id='producao'></a>
## 10. Produtização (*Productionalization*)

Colocar o modelo **dentro do produto**:

1. **API ou batch** que gera predições em escala
2. **Monitoramento:** drift, queda de acurácia, volume anomalias
3. **Retreino agendado** + versionamento de modelos
4. **Fallback** se o modelo falhar (regra default, cache)
5. **Documentação** para engenharia, produto e analytics

```
  Modelo treinado
       ↓
  API REST / serviço
       ↓
  A/B test → rollout gradual
       ↓
  Monitoramento contínuo → retreino
```

---
<a id='templates'></a>
## 11. Templates e Resumo Interativo

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

# ── Checklist interativo ────────────────────────────────────────
print("CHECKLISTS DO PROCESSO DE MODELAGEM\n")

checklists = {
    "📋 Enquadramento": [
        "Qual problema estamos resolvendo (para quem)?",
        "O que é sucesso? (métrica + meta)",
        "Qual o impacto de negócio?",
        "Qual a urgência e escala?",
        "Quais restrições e dependências?",
        "Problema quantitativo, específico e focado no usuário?",
    ],
    "🔧 Solução": [
        "Tipo de algoritmo definido (com baseline)?",
        "Supervisionado ou não supervisionado — justificado?",
        "Classificação ou regressão — prós e contras documentados?",
        "Matriz esforço × impacto avaliada?",
        "Accuracy score e business score definidos?",
    ],
    "📊 Dados": [
        "Fontes mapeadas (tabelas, período, granularidade)?",
        "Dados em estado aceitável?",
        "Features e sinais suficientes?",
        "Precisa coletar mais dados?",
        "Auditoria feita (SQL, correlações)?",
    ],
    "🚀 Produtização": [
        "API ou batch que serve predições em escala?",
        "Monitoramento de drift configurado?",
        "Retreino agendado?",
        "Fallback definido?",
        "Documentação entregue?",
    ],
}

all_checks = {}
tab = widgets.Tab()
children = []

for secao, itens in checklists.items():
    checkboxes = [widgets.Checkbox(value=False, description=item, layout=widgets.Layout(width='95%')) for item in itens]
    all_checks[secao] = checkboxes
    box = widgets.VBox(checkboxes)
    children.append(box)

tab.children = children
for i, titulo in enumerate(checklists):
    tab.set_title(i, titulo)

display(tab)

In [ ]:
# Template: Plano de Projeto
from IPython.display import display, Markdown

display(Markdown("""
## Template — Plano de Projeto ML

| Item | Definição |
|---|---|
| **Problema / KPI** | *(preencher)* |
| **Usuário final** | *(preencher)* |
| **Intervalo de dados** | *(preencher)* |
| **Volume de dados** | *(preencher)* |
| **Features** | *(preencher)* |
| **Modelos a testar** | *(preencher)* |
| **Critério de modelo vencedor** | *(preencher)* |
| **Como servir em produção** | *(preencher)* |
| **Baseline atual** | *(preencher)* |
| **Lift esperado** | *(preencher)* |
"""))

In [ ]:
# Template: Registro de Experimentos
import pandas as pd

experimentos = pd.DataFrame({
    'Data':        ['', '', ''],
    'Experimento': ['', '', ''],
    'Features':    ['', '', ''],
    'Modelo':      ['', '', ''],
    'Métrica':     ['', '', ''],
    'Resultado':   ['', '', ''],
    'Próximo passo': ['', '', ''],
})

print("Template de registro de experimentos:")
experimentos

---
## Suas notas

- 
- 